In [11]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss

In [12]:
matches = pd.read_csv("../data/processed/features_v4.csv")

matches = matches.sort_values("MatchDateTime").reset_index(drop=True)

matches["Season"].unique()

<StringArray>
['21-22', '22-23', '23-24', '24-25', '25-26']
Length: 5, dtype: str

In [13]:
features = [
    "PPGDiff",
    "GoalAgainstDiffLast5",
    "ShotOTDiffLast5",
    "GDPerGameDiff",
    "GoalsAgainstPerGameDiff",
    "EloDiff"
]

In [14]:
seasons = [
    "22-23",
    "23-24",
    "24-25",
    "25-26"
]

In [15]:
rf_results = []

for test_season in seasons:

    train = matches[matches["Season"] < test_season]
    test = matches[matches["Season"] == test_season]

    X_train = train[features]
    y_train = train["FTR"]

    X_test = test[features]
    y_test = test["FTR"]

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)

    accuracy = accuracy_score(y_test, preds)
    loss = log_loss(y_test, probs, labels=model.classes_)

    rf_results.append({
        "Season": test_season,
        "Accuracy": accuracy,
        "LogLoss": loss,
        "Matches": len(test)
    })

rf_results = pd.DataFrame(rf_results)

rf_results

,Season,Accuracy,LogLoss,Matches
0,22-23,0.526316,1.006618,380
1,23-24,0.563158,0.953869,380
2,24-25,0.531579,0.975804,380
3,25-26,0.492105,1.034356,380


In [16]:
season_order = {
    "21-22": 0,
    "22-23": 1,
    "23-24": 2,
    "24-25": 3,
    "25-26": 4
}

matches["SeasonOrder"] = matches["Season"].map(season_order)

C:\Users\harry\AppData\Local\Temp\ipykernel_10392\3487263237.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["SeasonOrder"] = matches["Season"].map(season_order)


In [17]:
train = matches[matches["SeasonOrder"] < season_order[test_season]]

In [18]:
print("Average accuracy:", rf_results["Accuracy"].mean())
print("Average log loss:", rf_results["LogLoss"].mean())

Average accuracy: 0.5282894736842105
Average log loss: 0.9926618433402798


In [19]:
model_results = []

In [20]:
models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

In [21]:
for model_name, model in models.items():

    for test_season in seasons:

        train = matches[
            matches["SeasonOrder"] < season_order[test_season]
        ]

        test = matches[
            matches["Season"] == test_season
        ]

        X_train = train[features]
        y_train = train["FTR"]

        X_test = test[features]
        y_test = test["FTR"]

        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)

        model_results.append({
            "Model": model_name,
            "Season": test_season,
            "Accuracy": accuracy_score(y_test, preds),
            "LogLoss": log_loss(
                y_test,
                probs,
                labels=model.classes_
            )
        })

In [22]:
model_results = pd.DataFrame(model_results)

model_results

,Model,Season,Accuracy,LogLoss
0,Logistic Regression,22-23,0.550000,1.002831
1,Logistic Regression,23-24,0.571053,0.935190
2,Logistic Regression,24-25,0.536842,0.995096
3,Logistic Regression,25-26,0.484211,1.035984
4,Random Forest,22-23,0.526316,1.006618
5,Random Forest,23-24,0.563158,0.953869
6,Random Forest,24-25,0.531579,0.975804
7,Random Forest,25-26,0.492105,1.034356
8,Gradient Boosting,22-23,0.513158,1.080750
9,Gradient Boosting,23-24,0.552632,0.992336


In [23]:
model_results.groupby("Model")[["Accuracy", "LogLoss"]].mean()

,Accuracy,LogLoss
Model,,
Gradient Boosting,0.523684,1.024316
Logistic Regression,0.535526,0.992275
Random Forest,0.528289,0.992662
